# In the paper, we reported 4 configurations:

1. State Estimation Evaluation: `ANOMALY_DATASET=False, SWAP_TRAIN_TEST=False, TASKS=['peg_pick', 'probe', 'wrap']`
2. State Estimation Evaluation with extended train set: `ANOMALY_DATASET=False, SWAP_TRAIN_TEST=True, TASKS=['peg_pick', 'probe', 'wrap']`
3. Anomaly Detection: `ANOMALY_DATASET=True, SWAP_TRAIN_TEST=False, TASKS=['peg_pick', 'probe', 'wrap']`
4. Growing classes experiment: `ANOMALY_DATASET=False, SWAP_TRAIN_TEST=False, TASKS=['additional']`

- You can choose the decider models for eval. from choices
- Uncomment `ipt.show(small=True)` to plot confusion matrix for each train instance
- Uncomment `d_train.showcase_aligned(scale=1.0)` to see dataset imgs aligned by timestep for each train instance 
- Uncomment `d_test.showcase_aligned(scale=1.0)` to see dataset imgs aligned by timestep for each train instance
- Uncomment `user_study_plot_hist(...)` to generate histogram plot.
- `visualize_accuracies(..., jupyter_plot=True)` to plot here in notebook
- `visualize_accuracies(..., jupyter_plot=False)` to save the figures in folder 

In [ ]:
import numpy as np
from tqdm import tqdm
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels, user_study_tasks_only, user_study_nice_model_names, y_cls_to_nice_name, user_study_plot_hist, set_seed
from nocode_robot_programming.state_decision.state_decider import StateDeciderBase, DINOFeaturePresence, DINOFeaturePresenceConcat, DINOFeaturePresenceAttnGated, DINOWithMIL, StateDeciderSIFT, AEGP
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda
from nocode_robot_programming.state_decision.plots.benchmark_plot import visualize_accuracies
from nocode_robot_programming.state_decision.plots import pretty_confusion_matrix
from nocode_robot_programming.state_decision_dataset_prepare.dataset_auto import TrajectoryDatasetEvaluationViewBuilder
kill_other_ipykernels(force=True)
set_seed()
# First run creates/syncs this file in trajectory_data/trajectories.
# Edit use=0 and discard_reason there to remove bad trajectories from the dataset.
CRITERIA_CSV = "trajectory_criteria.csv"
dataset_builder = TrajectoryDatasetEvaluationViewBuilder(
    criteria_csv=CRITERIA_CSV,
    sync_criteria_csv=True,
    print_criteria_report=True,
    # discard_criteria = {"distractor"},
    # discard_criteria = {"distractor", "target_features_not_visible_or_too_small", "teaching_multiple_things_at_once", "hand_in_scene",  "camera_not_working", "very_bad_train_data"},
    # discard_criteria = {"target_features_not_visible_or_too_small", "teaching_multiple_things_at_once", "hand_in_scene",  "camera_not_working", "very_bad_train_data"},
    discard_criteria = {},
)

ANOMALY_DATASET = False
SWAP_TRAIN_TEST = True
TASKS = ['peg_pick', 'probe', 'wrap']
# TASKS = ['test']
out_dir = f'{"anom" if ANOMALY_DATASET else "sest"}_{"stt" if SWAP_TRAIN_TEST else ""}_{len(TASKS)}'

if ANOMALY_DATASET:
    percentile_keep = 0.1
else:
    percentile_keep = None

train_stats, test_stats, model_names = [], [], []
for decider in [
        # DINOFeaturePresence(dino_variant= "dinov2_vits14", percentile_keep=percentile_keep), # best model
        # DINOFeaturePresence(dino_variant= "facebook/dinov3-vits16-pretrain-lvd1689m", percentile_keep=percentile_keep), # DINOv3
        # DINOFeaturePresenceConcat(dino_variant= "dinov2_vits14", percentile_keep=percentile_keep),
        DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=percentile_keep, attn_mode = "hard", head_reduce = "mean", attn_keep = 0.4),
        # DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=percentile_keep, attn_mode = "hard", head_reduce = "max" , attn_keep = 0.4),
        # DINOWithMIL(dino_variant= "dinov2_vits14", percentile_keep=percentile_keep, att_hidden = 128, dropout_p = 0.1, lr = 1e-4, weight_decay = 1e-3, epochs = 1000),
        # DINOFeaturePresence(percentile_keep=percentile_keep, dino_variant="dinov2_vitg14"),
        # DINOFeaturePresence(dino_variant = "dinov2_vitl14", percent_keep=percentile_keep),
        # StateDeciderSIFT(method = "SIFT", anomaly_percentile=percentile_keep), # suggesting the background problem
        # StateDeciderSIFT(method = "AKAZE"),
        # StateDeciderSIFT(method = "ORB"),
        # AEGP(binary=False, pix=64),
        # AEGP(binary=False, pix=224, crop=True),
    ]:
    train_stats_model, test_stats_model, dataset_names, stats = [], [], [], {}
    for name in (pbar := tqdm(user_study_tasks_only(dataset_builder, TASKS))):
        for d1, d2, d_text in dataset_builder.load_eval_from_task(name, anomaly=ANOMALY_DATASET): # I swapped train and test here in purpose!
            if not SWAP_TRAIN_TEST: 
                d_train, d_test = d1, d2
            else:
                d_train, d_test = d2, d1
            pbar.set_description(f"Model: {decider}, Dataset: {d_text}")
            
            try:
                decider.train(d_train.X, d_train.y_int, d_train.y_cls); ipt.save() # loss fig obtained in train function
            except AssertionError:
                continue

            y_pred = decider.predict_many(d_train.X)
            pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"train,{decider.short_name}", show=False); ipt.save()
            train_stats_model.append((np.array(d_train.y_names) == np.array(y_pred)).mean())

            y_pred = decider.predict_many(d_test.X)
            pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"test,{decider.short_name}", show=False); ipt.save()
            test_stats_model.append((np.array(d_test.y_names) == np.array(y_pred)).mean())
            
            # ipt.show(small=True) # See confusion matrices
            # d_train.showcase_aligned(scale=1.0) # See dataset imgs aligned by timestep
            # d_test.showcase_aligned(scale=1.0) # See dataset imgs aligned by timestep
            # ipt.delete() # Don't plot
            dataset_names.append(d_text.split(",")[0])

            stats[d_text] = (np.array(d_test.y_names) == np.array(y_pred)).mean()
    train_stats.append(train_stats_model)
    test_stats.append(test_stats_model)
    model_names.append(decider.short_name)
    # user_study_plot_hist(stats, [[0.2, 0.7, "*", 0.3], [0.7, 0.85, "**", 0.5]], f"sns_hist_2_{decider.short_name}.pdf", print_examples = 0, print_howmany_over_90 = True, bins = 21, folder=out_dir)

visualize_accuracies(100 * np.array(train_stats), 100 * np.array(test_stats), model_names, dataset_names, out_dir=out_dir, jupyter_plot=False)
f"Overall accuracy on train data: {round(100 * np.array(train_stats).mean())}% and test data {round(100 * np.array(test_stats).mean())}%"

### Quick fix for `SWAP_TRAIN_TEST` option where a single DS fails

- In the dataset, there was one skill part that missed executions
- TODO: Fix dataset

In [ ]:
from copy import deepcopy
train_stats2 = deepcopy(train_stats)
test_stats2 = deepcopy(test_stats)
test_stats2[-1] = test_stats2[-1][:-1]
test_stats2[-2] = test_stats2[-2][:-1]
train_stats2[-1] = train_stats2[-1][:-1]
train_stats2[-2] = train_stats2[-2][:-1]
dataset_names2 = dataset_names[:-1]
visualize_accuracies(100 * np.array(train_stats2), 100 * np.array(test_stats2), model_names, dataset_names2, out_dir=out_dir, jupyter_plot=False)